In [1]:
import scanpy as sc
import anndata as ad
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
from matplotlib.colors import ListedColormap, rgb2hex
import numpy as np
import warnings
import pandas as pd
warnings.filterwarnings('ignore')
import numpy as np
from sklearn.metrics import jaccard_score
import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams['pdf.fonttype'] = 42 # ADOBE AI 字帖

from matplotlib.font_manager import fontManager, FontProperties

fontManager.addfont('/data/work/Arial.ttf')

font = FontProperties(fname='/data/work/Arial.ttf')
font_name = font.get_name()
plt.rcParams['font.family'] = font_name

In [2]:
mouse = sc.read_h5ad('/data/work/22.fusemap/01.datas/99.all/mouse_brain_stereo_3d.h5ad')
dic = dict(zip(mouse.obs['area_name'], mouse.obs['area_name_0']))

In [3]:
adata = sc.read_h5ad('/data/work/22.fusemap/05.stereoalign/02.scvi/99.all/scvi_corrected.h5ad')
adata.obs['region'] = [i if i not in dic.keys() else dic[i] for i in adata.obs['region'] ]

In [4]:
lamprey_csv = pd.read_csv('/data/work/22.fusemap/05.stereoalign/untitled.txt', sep = '\t')
dic = dict(zip(lamprey_csv['Brain subregion'], lamprey_csv['Brain region']))
adata.obs['region'] = [i if i not in dic.keys() else dic[i] for i in adata.obs['region'] ]

In [5]:
dic = {
    '1': 'mouse',
    '0': 'lamprey',
}
adata.obs['species'] = [dic[i] for i in adata.obs['species']]
adata.obs['region'] = adata.obs['species'].astype(str) +'_'+ adata.obs['region'].astype(str)

In [6]:
scvi_repr = adata.obsm['aligned_scvi']
cell_types = adata.obs['region'].values

In [7]:
df = pd.DataFrame(scvi_repr, index=adata.obs.index)
df['cell_type'] = cell_types
mean_repr = df.groupby('cell_type').mean()

In [8]:
raw_columns = ['lamprey_Olfactory Bulb',  'lamprey_Pallium', 
               'lamprey_Primordium hippocampi', 
               'lamprey_Striatum',
               'lamprey_Hypothalamus', 
               'lamprey_Pineal gland','lamprey_Thalamus',
               'lamprey_Habenula',
               'lamprey_Midbrain',
               'lamprey_Cerebellum-like',
               'lamprey_Rhombencephalon',
               'lamprey_Choroid plexus', 'lamprey_Meninges', 'lamprey_Ependyma',]
raw_indexs = ['mouse_Olfactory areas', 'mouse_Isocortex',
              'mouse_Cortical subplate', 'mouse_Hippocampal',
              'mouse_Hypothalamus', 'mouse_Thalamus',
              'mouse_Midbrain',
              'mouse_Cerebellum', 'mouse_Cerebral nuclei',
              'mouse_Medulla', 'mouse_Pons',
              'mouse_ventricular systems']

In [11]:
corr_matrix = mean_repr.T.corr('pearson')
corr_matrix = corr_matrix[raw_columns].loc[raw_indexs]
corr_matrix.columns = [i.replace('lamprey_', '') for i in raw_columns]
corr_matrix.columns = [i.replace('_', ' ') for i in corr_matrix.columns]
corr_matrix.index = [i.replace('mouse_', '') for i in raw_indexs]
corr_plot2(corr_matrix, '/data/work/22.fusemap/05.stereoalign/02.scvi/99.all/scvi_person.pdf')

In [10]:






import matplotlib.pyplot as plt
import seaborn as sns
def corr_plot2(corr_matrix, save_name):


    g = sns.heatmap(
        corr_matrix,
        annot=False,
        cmap='coolwarm',
        square=True,
        linewidths=.5,
        # row_cluster=True,
        # col_cluster=True,
        yticklabels=True,
    )

    g.set_xlabel('Lamprey Brain Region', fontsize=14, fontweight='bold')
    g.set_ylabel('Mouse Brain Region', fontsize=14, fontweight='bold')

    plt.setp(g.get_xticklabels(), rotation=45, ha='right')
    plt.setp(g.get_yticklabels(), rotation=0)

    plt.savefig(save_name, bbox_inches = 'tight')
    plt.close()
    # plt.show()

In [12]:






import matplotlib.pyplot as plt
import seaborn as sns
def corr_plot(corr_matrix, save_name):


    g = sns.clustermap(
        corr_matrix,
        annot=False,
        cmap='coolwarm',
        square=True,
        linewidths=.5,
        cbar_pos=None,          
        dendrogram_ratio=0.15,  
        figsize=(9, 6),
        row_cluster=True,
        col_cluster=True,
        yticklabels=True,
        tree_kws={'linewidths': 1.0}
    )

    g.ax_heatmap.yaxis.tick_left()
    g.ax_heatmap.yaxis.set_label_position('left')

    heatmap_pos = g.ax_heatmap.get_position()
    row_dend_pos = g.ax_row_dendrogram.get_position()

    g.ax_row_dendrogram.set_position([heatmap_pos.x1,
                                      row_dend_pos.y0, 
                                     row_dend_pos.width, row_dend_pos.height])

    g.ax_row_dendrogram.invert_xaxis()

    cbar_ax = g.fig.add_axes([heatmap_pos.x1 + 0.2, 0.5, 0.03, 0.2])  
    cbar_ax.set_title('Correlation', fontsize=12, pad=10)
    g.fig.colorbar(g.ax_heatmap.collections[0], cax=cbar_ax)

    g.ax_heatmap.set_xlabel('Lamprey Brain Region', fontsize=14, fontweight='bold')
    g.ax_heatmap.set_ylabel('Mouse Brain Region', fontsize=14, fontweight='bold')

    plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right')
    plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0)

    plt.savefig(save_name, bbox_inches = 'tight')
    plt.close()

In [11]:
corr_matrix = mean_repr.T.corr('pearson')
corr_matrix = corr_matrix[raw_columns].loc[raw_indexs]
corr_matrix.columns = [i.replace('lamprey_', '') for i in raw_columns]
corr_matrix.columns = [i.replace('_', ' ') for i in corr_matrix.columns]
corr_matrix.index = [i.replace('mouse_', '') for i in raw_indexs]
corr_plot(corr_matrix, '/data/work/22.fusemap/05.stereoalign/02.scvi/99.all/scvi_person.pdf')

In [13]:
corr_matrix = mean_repr.T.corr('spearman')

corr_matrix = corr_matrix[raw_columns].loc[raw_indexs]
corr_matrix.columns = [i.replace('lamprey_', '') for i in raw_columns]
corr_matrix.columns = [i.replace('_', ' ') for i in corr_matrix.columns]
corr_matrix.index = [i.replace('mouse_', '') for i in raw_indexs]
corr_plot2(corr_matrix, '/data/work/22.fusemap/05.stereoalign/02.scvi/99.all/scvi_spearman.pdf')

In [14]:
corr_matrix = mean_repr.T.corr('kendall')

corr_matrix = corr_matrix[raw_columns].loc[raw_indexs]
corr_matrix.columns = [i.replace('lamprey_', '') for i in raw_columns]
corr_matrix.columns = [i.replace('_', ' ') for i in corr_matrix.columns]
corr_matrix.index = [i.replace('mouse_', '') for i in raw_indexs]
corr_plot2(corr_matrix, '/data/work/22.fusemap/05.stereoalign/02.scvi/99.all/scvi_kendall.pdf')

In [16]:
raw_columns = ['lamprey_Olfactory Bulb',  'lamprey_Pallium', 
               'lamprey_Pineal gland',
               'lamprey_Striatum',
               'lamprey_Hypothalamus', 
               'lamprey_Thalamus',
               'lamprey_Habenula',
               'lamprey_Cerebellum-like',
               'lamprey_Midbrain',
               
               'lamprey_Primordium hippocampi', 'lamprey_Rhombencephalon',
               # 'lamprey_Choroid plexus', 'lamprey_Meninges', 'lamprey_Ependyma',
              ]
raw_indexs = ['mouse_Olfactory areas', 'mouse_Isocortex',
              'mouse_Cortical subplate', 'mouse_Hippocampal',
              'mouse_Hypothalamus', 'mouse_Thalamus',
              'mouse_Midbrain',
              'mouse_Cerebral nuclei',
              'mouse_Medulla', 'mouse_Pons',
              # 'mouse_ventricular systems'
              
              'mouse_Cerebellum', 
              
             ]



In [ ]:
corr_matrix = mean_repr.T.corr('pearson')
corr_matrix = corr_matrix[raw_columns].loc[raw_indexs]
corr_matrix.columns = [i.replace('lamprey_', '') for i in raw_columns]
corr_matrix.columns = [i.replace('_', ' ') for i in corr_matrix.columns]
corr_matrix.index = [i.replace('mouse_', '') for i in raw_indexs]
g = sns.heatmap(
    corr_matrix,
    annot=False,
    cmap='coolwarm',
    square=True,
    linewidths=.5,
    # row_cluster=True,
    # col_cluster=True,
    yticklabels=True,
)

g.set_xlabel('Lamprey Brain Region', fontsize=14, fontweight='bold')
g.set_ylabel('Mouse Brain Region', fontsize=14, fontweight='bold')

plt.setp(g.get_xticklabels(), rotation=45, ha='right')
plt.setp(g.get_yticklabels(), rotation=0)
# plt.show()
plt.savefig('/data/work/22.fusemap/05.stereoalign/02.scvi/99.all/scvi_20251031/pearson.pdf', bbox_inches = 'tight')
plt.close()

In [22]:
corr_matrix = mean_repr.T.corr('kendall')
corr_matrix = corr_matrix[raw_columns].loc[raw_indexs]
corr_matrix.columns = [i.replace('lamprey_', '') for i in raw_columns]
corr_matrix.columns = [i.replace('_', ' ') for i in corr_matrix.columns]
corr_matrix.index = [i.replace('mouse_', '') for i in raw_indexs]
g = sns.heatmap(
    corr_matrix,
    annot=False,
    cmap='coolwarm',
    square=True,
    linewidths=.5,
    # row_cluster=True,
    # col_cluster=True,
    yticklabels=True,
)

g.set_xlabel('Lamprey Brain Region', fontsize=14, fontweight='bold')
g.set_ylabel('Mouse Brain Region', fontsize=14, fontweight='bold')

plt.setp(g.get_xticklabels(), rotation=45, ha='right')
plt.setp(g.get_yticklabels(), rotation=0)
# plt.show()
plt.savefig('/data/work/22.fusemap/05.stereoalign/02.scvi/99.all/scvi_20251031/kendall.pdf', bbox_inches = 'tight')
plt.close()

In [21]:
corr_matrix = mean_repr.T.corr('spearman')
corr_matrix = corr_matrix[raw_columns].loc[raw_indexs]
corr_matrix.columns = [i.replace('lamprey_', '') for i in raw_columns]
corr_matrix.columns = [i.replace('_', ' ') for i in corr_matrix.columns]
corr_matrix.index = [i.replace('mouse_', '') for i in raw_indexs]
g = sns.heatmap(
    corr_matrix,
    annot=False,
    cmap='coolwarm',
    square=True,
    linewidths=.5,
    # row_cluster=True,
    # col_cluster=True,
    yticklabels=True,
)

g.set_xlabel('Lamprey Brain Region', fontsize=14, fontweight='bold')
g.set_ylabel('Mouse Brain Region', fontsize=14, fontweight='bold')

plt.setp(g.get_xticklabels(), rotation=45, ha='right')
plt.setp(g.get_yticklabels(), rotation=0)
# plt.show()
plt.savefig('/data/work/22.fusemap/05.stereoalign/02.scvi/99.all/scvi_20251031/spearman.pdf', bbox_inches = 'tight')
plt.close()